In [5]:
import pandas as pd
import numpy as np
import json
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

# 1. LOAD DATA

In [6]:
df = pd.read_csv('Student_Performance.csv')

# 2. PREPROCESSING

In [7]:
# Map ordinal text to numbers
ordinal_mapping = {
    'Excellent': 3,
    'Vg': 2,
    'Good': 1,
    'Average': 0
}

time_mapping = {
    'ONE': 1, 'TWO': 2, 'THREE': 3, 'FOUR': 4, 'FIVE': 5, 'SIX': 6, 'SEVEN': 7
}

# Apply mappings — astype(str) normalises pandas StringDtype to plain object dtype
df['Class_X_Percentage'] = df['Class_X_Percentage'].astype(str).map(ordinal_mapping)
df['Class_XII_Percentage'] = df['Class_XII_Percentage'].astype(str).map(ordinal_mapping)
df['Performance'] = df['Performance'].astype(str).map(ordinal_mapping)
df['time'] = df['time'].astype(str).map(time_mapping)

# Feature engineering — combined academic scores as extra signals
df['academic_sum']  = df['Class_X_Percentage'] + df['Class_XII_Percentage']
df['academic_prod'] = df['Class_X_Percentage'] * df['Class_XII_Percentage']

# One-Hot Encode nominal columns (Gender, Caste, etc.)
categorical_cols = ['Gender', 'Caste', 'coaching', 'Class_ten_education', 
                    'twelve_education', 'medium', 'Father_occupation', 'Mother_occupation']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Define Features (X) and Target (y)
X = df_encoded.drop('Performance', axis=1)
y = df_encoded['Performance']

# Split Data (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. TRAIN MODEL

In [8]:
# Slow-learning GradientBoostingClassifier — many shallow trees + small lr
# consistently outperforms RF and XGBoost on small tabular datasets
model = GradientBoostingClassifier(
    n_estimators=800,
    learning_rate=0.02,
    max_depth=3,
    min_samples_leaf=5,
    subsample=0.8,
    max_features="sqrt",
    random_state=42,
)

model.fit(X_train, y_train)

cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
print(f"CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Save the model for the FastAPI backend
joblib.dump(model, 'student_performance_model.pkl')
print("Model saved as 'student_performance_model.pkl'")

CV accuracy: 0.4794 ± 0.0370
Model saved as 'student_performance_model.pkl'


# 4. EVALUATION

In [9]:
y_pred = model.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")

# Generate evaluation JSON
evaluation_json = {
    "model_name": "GradientBoostingClassifier (slow-learning, feature-engineered)",
    "dataset_source": "Proxy Dataset (Kaggle)",
    "metrics": {
        "accuracy": float(f"{accuracy:.4f}"),
        "f1_score": float(f"{f1:.4f}"),
        "precision": float(f"{precision:.4f}"),
        "recall": float(f"{recall:.4f}")
    },
    "confusion_matrix": conf_matrix.tolist(),
    "class_labels": {v: k for k, v in ordinal_mapping.items()}
}

with open('evaluation_json.json', 'w') as f:
    json.dump(evaluation_json, f, indent=4)

print("Evaluation saved to 'evaluation_json.json'")

Accuracy : 0.5075
F1 Score : 0.4989
Precision: 0.5729
Recall   : 0.5075
Evaluation saved to 'evaluation_json.json'
